<a href="https://www.kaggle.com/code/shravankumarpandey/next-word-prediction-using-lstm?scriptVersionId=324719813" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shravankumarpandey/large-scale-english-text-dataset/lstm_next_word_corpus.txt


In [2]:
with open("/kaggle/input/datasets/shravankumarpandey/large-scale-english-text-dataset/lstm_next_word_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Importing Libraries

In [3]:
import tensorflow 
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

2026-06-05 10:01:16.909786: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780653677.121631      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780653677.175988      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780653677.646390      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780653677.646481      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780653677.646485      23 computation_placer.cc:177] computation placer alr

# Initialize Tokenizer

In [4]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])

In [5]:
tokenizer.word_index

{'and': 1,
 'to': 2,
 'in': 3,
 'many': 4,
 'that': 5,
 'being': 6,
 'people': 7,
 'new': 8,
 'influence': 9,
 'often': 10,
 'discussions': 11,
 'about': 12,
 'experts': 13,
 'emphasize': 14,
 'the': 15,
 'importance': 16,
 'of': 17,
 'observation': 18,
 'learning': 19,
 'adaptation': 20,
 'artificial': 21,
 'intelligence': 22,
 'technology': 23,
 'business': 24,
 'education': 25,
 'travel': 26,
 'daily': 27,
 'athletes': 28,
 'improve': 29,
 'performance': 30,
 'through': 31,
 'training': 32,
 'recovery': 33,
 'strategic': 34,
 'decision': 35,
 'making': 36,
 'economic': 37,
 'conditions': 38,
 'can': 39,
 'affect': 40,
 'employment': 41,
 'opportunities': 42,
 'investment': 43,
 'decisions': 44,
 'consumer': 45,
 'behavior': 46,
 'is': 47,
 'applied': 48,
 'solve': 49,
 'problems': 50,
 'healthcare': 51,
 'finance': 52,
 'researchers': 53,
 'study': 54,
 'complex': 55,
 'systems': 56,
 'understand': 57,
 'patterns': 58,
 'appear': 59,
 'nature': 60,
 'society': 61,
 'modern': 62,
 'c

In [6]:
l=len(tokenizer.word_index)
print(l)

161


In [7]:
for sentence in text.split("\n"):
    print(sentence)

In discussions about technology, many experts emphasize the importance of observation, learning, and adaptation. Creative thinking helps people adapt to changing circumstances and unexpected challenges. A successful business usually balances innovation, customer satisfaction, and long term planning. Daily routines may seem ordinary, yet they influence productivity and personal growth. People often learn new skills by practicing consistently and reflecting on their experiences. Travel exposes individuals to different cultures, languages, and perspectives. Modern technology continues to influence communication, transportation, and entertainment in many ways. Students benefit from curiosity, discipline, and access to reliable information sources. Researchers study complex systems to understand patterns that appear in nature and society.

In discussions about sports, many experts emphasize the importance of observation, learning, and adaptation. Historical events often shape political inst

In [8]:
input_subsequences = []

for sentence in text.split("\n"):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        input_subsequences.append(tokenized_sentence[:i+1])  

In [9]:
max_len=max([len(x) for x in input_subsequences])
print(max_len)

114


# Padding

In [10]:
padded_input_sequences=pad_sequences(input_subsequences,maxlen=max_len,padding="pre")
print(padded_input_sequences)

[[  0   0   0 ...   0   3  11]
 [  0   0   0 ...   3  11  12]
 [  0   0   0 ...  11  12  23]
 ...
 [  0   0   0 ... 152   2 153]
 [  0   0   0 ...   2 153 154]
 [  0   0   0 ... 153 154 155]]


In [11]:
X=padded_input_sequences[:,:-1]
y=padded_input_sequences[:,-1]

In [12]:
X.shape

(51550, 113)

In [13]:
y.shape

(51550,)

# One Hot Encoding

In [14]:
y=to_categorical(y,num_classes=l+1)

# Model Building

In [15]:
model= Sequential()
model.add(Embedding(l+1,100,input_length=max_len))
model.add(LSTM(150))
model.add(Dense(l+1,activation="softmax"))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1780653693.438479      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Model Compilation

In [16]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# Model Fitting

In [17]:
model.fit(
    X,
    y,
    epochs=25
)

Epoch 1/25


I0000 00:00:1780653697.390974      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


1611/1611 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - accuracy: 0.7872 - loss: 1.0097
Epoch 2/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.9212 - loss: 0.2426
Epoch 3/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9206 - loss: 0.2363
Epoch 4/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9207 - loss: 0.2339
Epoch 5/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9210 - loss: 0.2318
Epoch 6/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9208 - loss: 0.2313
Epoch 7/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9211 - loss: 0.2304
Epoch 8/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9212 - loss: 0.2300
Epoch 9/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9214 - loss: 0.2290
Epoch 10/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9217 - loss: 0.2285
Epoch 11/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.9208 - loss: 0.2280
Epoch 12/25
1611/16

# Model Prediction

In [18]:
import numpy as np

def predict_next_word(text, num_words=10):
    for _ in range(num_words):

        token_text = tokenizer.texts_to_sequences([text])[0]

        padded_token = pad_sequences(
            [token_text],
            maxlen=max_len-1,
            padding="pre"
        )

        prediction = model.predict(padded_token, verbose=0)

        pos = np.argmax(prediction, axis=-1)[0]

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == pos:
                output_word = word
                break

        text = text + " " + output_word

    return text

In [19]:
seed_text = "artificial intelligence is"
generated_text = predict_next_word(seed_text, num_words=10)

print(generated_text)

artificial intelligence is discussions about business many experts emphasize the importance of observation
